In [1]:
# !pip install pymupdf
import pymupdf
# !pip install pandas
import pandas as pd

In [2]:
import os
import json
import csv

In [6]:
jrb_edition = 3
jrb_folderpath = os.path.join("..", "public", f"jrb-{jrb_edition}-ed")
print(jrb_folderpath) 
os.makedirs(jrb_folderpath, exist_ok=True)

../public/jrb-3-ed


In [7]:
jrb_filepath = os.path.join("..", "public", f"jrb_{jrb_edition}.pdf")
print(jrb_filepath)

song_name_page_nums = os.path.join("..", "data_files", f"jrb_{jrb_edition}_index.tsv")
print(song_name_page_nums)

../public/jrb_3.pdf
../data_files/jrb_3_index.tsv


In [9]:
df = pd.read_csv(song_name_page_nums, sep='\t', names=["page_num", "song_name"], quoting=csv.QUOTE_NONE)
df['page_num'] = pd.to_numeric(df['page_num'], errors='coerce')
df_sorted = df.sort_values(by='page_num')
df_sorted

,page_num,song_name
0,1,After You
1,2,After You've Gone
2,3,Ain't Misbehavin'
4,4,All Or Nothing At All
3,6,Alanjuneally
...,...,...
285,354,You Must Believe In Spring
287,355,You've Changed
286,356,You're Everything
288,358,Younger Than Springtime


In [10]:
tot = 0
for i in range(len(df_sorted)):
  current_row = df_sorted.iloc[i]
  page_scans = 1

  if i < len(df_sorted) - 1:
    next_row = df_sorted.iloc[i + 1]
    if not current_row["page_num"] + 1 == next_row["page_num"]:
      page_scans = next_row["page_num"] - current_row["page_num"]
      print(f"{current_row['page_num']:03} - {current_row['song_name']} - {page_scans}")


  tot += 1

print(tot)

004 - All Or Nothing At All - 2
010 - Armando's Rhumba - 2
018 - Be My Love - 2
020 - Beside Myself - 2
022 - Bess, You Is My Woman Now - 2
032 - Bop Shop - 2
038 - Bud Powell - 2
046 - Celia - 2
050 - Cheek To Cheek - 3
056 - Cool Eyes - 2
060 - Dacapolypso - 2
062 - Day In, Day Out - 2
064 - Dig - 2
068 - Don't Look Back - 2
076 - Endlessly - 2
080 - Ev'ry Time We Say Good Bye - 2
092 - From the Heart  - 2
098 - Gaviota - 2
104 - Harlem Nocturne - 2
106 - Heartsong - 2
108 - High Hopes - 2
110 - High Wire The Aerialist - 2
118 - I Concentrate On You - 2
122 - INeed YouHere - 2
124 - I Wish You Love - 2
126 - I'll Be Around - 2
136 - I've Got The World On A String - 2
138 - I've Got You Under My Skin - 2
146 - In the Days of Our Love - 2
152 - It's All Right With Me - 2
158 - Jitterbug Waltz - 2
160 - Just One Of Those Things - 2
164 - Ladies In Mercedes - 2
168 - Leap Of Faith - 2
172 - Little Face [Tab_] - 2
174 - Little Girl Blue - 2
176 - Long View, The - 2
182 - Love Walked In - 

In [11]:
def extract_and_save_pages(filename, pdf_save_path, start, end):
  doc = pymupdf.open(filename)
  pages_to_keep = list(range(start, end))
  doc.select(pages_to_keep) 
  doc.save(
      pdf_save_path + ".pdf",
      garbage=3, 
      deflate=True, 
      clean=True
  )
  doc.close()

In [13]:
tot = 0
offset = 5 # this has to be computed manually.
# For:
  # jrb_6, offset is -1
  # jrb_5, offset is 13
  # jrb_3, offset is 5

for i in range(len(df_sorted)):
  current_row = df_sorted.iloc[i]
  page_scans = 1

  if i < len(df_sorted) - 1:
    next_row = df_sorted.iloc[i + 1]
    if not current_row["page_num"] + 1 == next_row["page_num"]:
      page_scans = next_row["page_num"] - current_row["page_num"]
      

  song_folder = os.path.join(jrb_folderpath, str(current_row["page_num"]))
  # print(song_folder)
  # print(f"{current_row['page_num']:03} - {current_row['song_name']} - {page_scans}")
  cleaned_songname = current_row['song_name'].strip().replace(" ", "_")
  # print(cleaned_songname)

  os.makedirs(song_folder, exist_ok=True)
  savename = os.path.join(song_folder, cleaned_songname)
  pdf_start_page = current_row["page_num"] - 1 + offset # subtract 1 for 0 index, add offset
  pdf_end_page = pdf_start_page + page_scans
  if pdf_start_page == pdf_end_page: # happens because 2 songs are on 1 page. 
    pdf_end_page += 1
  # print(pdf_start_page, pdf_end_page, cleaned_songname)
  extract_and_save_pages(jrb_filepath, savename, pdf_start_page, pdf_end_page)

  tot += 1

print(tot)

290


In [14]:
# Extra util to create json files
df_sorted["edition"] = jrb_edition
save_filename = os.path.join("..", "data_files", f"jrb_{jrb_edition}_index.json")
df_sorted.to_json(save_filename, orient="records", indent=2)